# Replication Study — Synthetic Data Generation (full ε sweep)

Generates CTGAN + TVAE synthetic data at nodp + ε=10/5/1 for the 4
replication datasets (adult, churn_modelling, covertype, cardio) —
same epsilon sweep already run on diabetes_130us/acs_income, reusing
the corrected `dp_wrapper.py` unchanged.

32 fit/sample runs total: 2 generators × 4 datasets × 4 conditions
(nodp, 10, 5, 1).

Output: `data/synthetic/{dataset}/{generator}_nodp.csv` and
`{generator}_eps{1,5,10}.csv` — saved directly (no `new/` subfolder;
that convention was only needed to separate pre-fix/post-fix runs on
diabetes_130us/acs_income, which doesn't apply here since these are
fresh runs with the already-corrected wrapper).

**Interface note**: uses the same `generator.fit()` / `generator.model.sample()`
/ `generator.save_synthetic()` pattern for nodp, and `DPWrapper(generator,
epsilon, config).fit()` / `.sample()` / `.save_synthetic()` for the DP runs —
identical to how diabetes_130us/acs_income were generated. Not re-verified
against your real generator classes in this session (see prior message);
the dataset/epsilon loop structure is the reusable part if any method name
needs adjusting.

In [ ]:
import os
os.chdir('..')

import json
import yaml
import pandas as pd
from pathlib import Path

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.generators.ctgan_generator import CTGANGenerator
from src.generators.tvae_generator import TVAEGenerator
from src.generators.dp_wrapper import DPWrapper

REPLICATION_DATASETS = ['adult', 'churn_modelling', 'covertype', 'cardio']
GENERATOR_CLASSES = {'ctgan': CTGANGenerator, 'tvae': TVAEGenerator}
EPSILONS = config['differential_privacy']['epsilons']  # [1, 5, 10]

missing = [d for d in REPLICATION_DATASETS
           if not (Path(config['datasets'][d]['processed_dir']) / 'train.csv').exists()]
if missing:
    raise FileNotFoundError(
        f'{missing} have no processed train.csv — run 09_replication_ingestion.ipynb first.'
    )
print(f'All 4 replication datasets ready. Epsilons: {EPSILONS}')

In [ ]:
def generate_nodp(dataset_name, gen_name, gen_cls, config):
    """Fits and samples the plain (non-DP) generator, saves as '{gen}_nodp.csv'."""
    generator = gen_cls(config, dataset_name)
    train_df = generator.load_train_data()
    metadata = generator.build_metadata(train_df)

    generator.fit(train_df, metadata)

    n_samples = len(train_df)  # paper convention: synthetic size matches real data size
    synthetic_df = generator.model.sample(num_rows=n_samples)

    label = f'{gen_name}_nodp'
    generator.save_synthetic(synthetic_df, label)
    return label, n_samples


def generate_dp(dataset_name, gen_name, gen_cls, epsilon, config):
    """Fits and samples the DP-wrapped generator at the given epsilon,
    saves as '{gen}_eps{epsilon}.csv'."""
    generator = gen_cls(config, dataset_name)
    train_df = generator.load_train_data()
    metadata = generator.build_metadata(train_df)

    dp_gen = DPWrapper(generator, epsilon, config)
    dp_gen.fit(train_df, metadata)

    n_samples = len(train_df)
    synthetic_df = dp_gen.sample(n_samples)

    label = f'{gen_name}_eps{epsilon}'
    dp_gen.save_synthetic(synthetic_df, label)
    return label, n_samples


print('Generation functions defined.')

In [ ]:
results = []

for dataset_name in REPLICATION_DATASETS:
    print(f'\n{"="*60}\n  {dataset_name}\n{"="*60}')

    for gen_name, gen_cls in GENERATOR_CLASSES.items():

        # --- nodp ---
        print(f'\n  [{gen_name}_nodp]')
        try:
            label, n = generate_nodp(dataset_name, gen_name, gen_cls, config)
            results.append({'dataset': dataset_name, 'generator': gen_name, 'epsilon': 'nodp',
                             'status': 'OK', 'n_rows': n})
            print(f'    OK — {n} rows saved as {label}.csv')
        except Exception as e:
            results.append({'dataset': dataset_name, 'generator': gen_name, 'epsilon': 'nodp',
                             'status': f'ERROR: {e}', 'n_rows': None})
            print(f'    ERROR: {e}')

        # --- DP variants ---
        for epsilon in EPSILONS:
            label = f'{gen_name}_eps{epsilon}'
            print(f'\n  [{label}]')
            try:
                label, n = generate_dp(dataset_name, gen_name, gen_cls, epsilon, config)
                results.append({'dataset': dataset_name, 'generator': gen_name, 'epsilon': str(epsilon),
                                 'status': 'OK', 'n_rows': n})
                print(f'    OK — {n} rows saved as {label}.csv')
            except Exception as e:
                results.append({'dataset': dataset_name, 'generator': gen_name, 'epsilon': str(epsilon),
                                 'status': f'ERROR: {e}', 'n_rows': None})
                print(f'    ERROR: {e}')

results_df = pd.DataFrame(results)
print(f'\n\n{len(results_df)} runs attempted, {(results_df.status=="OK").sum()} succeeded.')
print(results_df.to_string(index=False))

## Sanity check generated synthetic data

Same cheap degeneracy check as before, now across all 32 combos — catch
constant columns or broken target distributions before the evaluation
notebook, not during it.

In [ ]:
eps_labels = ['nodp'] + [str(e) for e in EPSILONS]

for dataset_name in REPLICATION_DATASETS:
    target_col = config['datasets'][dataset_name]['target_col']
    for gen_name in GENERATOR_CLASSES:
        for eps in eps_labels:
            label = f'{gen_name}_nodp' if eps == 'nodp' else f'{gen_name}_eps{eps}'
            path = Path(config['datasets'][dataset_name]['synthetic_dir']) / f'{label}.csv'
            if not path.exists():
                continue
            df = pd.read_csv(path)
            n_constant = sum(df[c].nunique() <= 1 for c in df.columns)
            flag = '  <-- CHECK: constant columns present' if n_constant > 0 else ''
            print(f'{dataset_name}/{label}: shape={df.shape}, constant_cols={n_constant}{flag}')